# PyTorch ile Derin Öğrenme · Aktarım öğrenmesi ve dağıtım

Macademia · Büyükler için

> **Colab'da çalıştırma:** Bu dosyayı [colab.research.google.com](https://colab.research.google.com) adresinde **Dosya → Not defteri yükle** ile aç. GPU için **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç. Hücreleri yukarıdan aşağı sırayla çalıştır (Shift + Enter). PyTorch Colab'da kuruludur; bilgisayarında çalıştırmak istersen `pip install torch torchvision`.

In [ ]:
import torch, random, numpy as np
torch.manual_seed(0); random.seed(0); np.random.seed(0)          # tekrarlanabilirlik
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', device)

## 1. Önceden eğitilmiş ResNet-18

In [ ]:
from torch import nn
from torchvision import models
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
print(sum(p.numel() for p in model.parameters()))                   # 11 689 512
for name, mod in [('stem', nn.Sequential(model.conv1, model.bn1)), ('layer1', model.layer1), ('layer2', model.layer2), ('layer3', model.layer3), ('layer4', model.layer4)]:
    print(name, sum(p.numel() for p in mod.parameters()))

## 2. Dondur ve yeni baş ekle

In [ ]:
for p in model.parameters():
    p.requires_grad = False
model.fc = nn.Linear(512, 5)
print('eğitilen:', sum(p.numel() for p in model.parameters() if p.requires_grad))
for p in model.layer4.parameters():
    p.requires_grad = True                                           # ince ayar: son blok
print('layer4 çözüldü:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 3. Dışa aktarma: TorchScript ve ONNX

In [ ]:
model.eval()
ornek = torch.randn(1, 3, 224, 224)
ts = torch.jit.trace(model, ornek); ts.save('model_ts.pt')
torch.onnx.export(model, ornek, 'model.onnx', input_names=['goruntu'], output_names=['skor'], dynamic_axes={'goruntu': {0: 'N'}})
import os
print(os.path.getsize('model_ts.pt') // 1024, 'KB', os.path.getsize('model.onnx') // 1024, 'KB')

## 4. Dinamik int8 nicemleme

In [ ]:
import time
q = torch.ao.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)
x = torch.randn(8, 3, 224, 224)
for name, m in [('fp32', model), ('int8 (Linear)', q)]:
    t = time.perf_counter()
    with torch.no_grad():
        y = m(x)
    print(name, f'{(time.perf_counter() - t) * 1000:.0f} ms', y.shape)
torch.save(q.state_dict(), 'q.pt'); torch.save(model.state_dict(), 'f.pt')
print(os.path.getsize('f.pt') // 1024, 'KB ->', os.path.getsize('q.pt') // 1024, 'KB')

## Alıştırmalar
1. Kendi 5 sınıflı küçük görüntü kümenle (her sınıftan 50 görüntü) yalnız başı eğit.
2. `onnxruntime` ile ONNX modelini çalıştır ve çıktıyı PyTorch ile karşılaştır.
3. Statik nicemleme (kalibrasyonlu) dene.